# A2 Q1 - Click-History & Session Features

Produces `reranker_features.parquet` per dataset: one row per
`(impression_id, article_id)` pair over `article_ids_inview`, built from
behavioural signals already present in the unified schema
(`history.timestamp_sequence`/`read_time_sequence`/`scroll_percentage_sequence`,
`behaviors.session_id`) that Assignment 1 built but never consumed. This is
Stage 1 input for Q2's re-ranker -- see SPEC.md's `A2 Q1` section for the
full design (recency-weight formulas, why weighting is per-impression not
per-user, what's deliberately excluded as leakage, and the scale/checkpointing
discipline).

The same code also featurizes the two blind leaderboard populations
(`FEATURE_DATASETS=ebnerd_testset` / `mind_large_test`, SPEC.md A2 Q5 #6).
They have no `article_ids_clicked`, so `clicked` and
`clicks_earlier_in_session` are null there; their popularity basis (and, for
`ebnerd_testset`, the article catalog and embeddings) comes from the dataset
the re-ranker was trained on, and their rows are written in
(`user_id`, `impression_id`) order so `reranker_submission.ipynb` can align
its Stage-1 pass chunk-for-chunk.

Run top-to-bottom (or via `python feature_engineering.py`) to rebuild
`data/processed/{dataset}/reranker_features.parquet` +
`feature_metrics.json`. Requires `history.parquet`/`behaviors.parquet`/
`articles.parquet`/`article_embeddings.parquet` to already exist for every
dataset in scope.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.embeddings import weighted_mean_pool, mean_pool
from cs4406m26_assignment1c1.evaluation import train_popularity_lookup
from cs4406m26_assignment1c1.reranker import SUBMISSION_POPULATIONS, population_article_id, write_user_sorted_source


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
PROGRESS_LOG = ROOT / "build_progress.log"
CHECKPOINT_DIR = DATA_DIR / "_feature_checkpoints"
# Impressions per checkpointed chunk -- same discipline as evaluate_ranking/
# generate_predictions (SPEC.md A2 Q1 #9): a crash anywhere loses at most
# one partial chunk, a retry skips every already-done chunk.
CHUNK_SIZE = 200_000

HALF_LIFE_HOURS = 72.0   # EB-NeRD recency decay, elapsed-time basis (SPEC.md A2 Q1 #3)
HALF_LIFE_CLICKS = 5.0   # MIND recency decay, ordinal-proxy basis (no per-click timestamps ever exist)
TRAIN_SAMPLE_CAP = 2_000_000
TRAIN_SAMPLE_SEED = 0
SPLITS = ["train", "val", "test"]
# The per-row key. `impression_id` alone is NOT unique on ebnerd_testset: its
# 200,000 beyond-accuracy rows all carry the sentinel impression_id 0 (A1
# SPEC.md Q5 #8 records the same fact, hit twice there). Joined on
# impression_id alone, the session-feature attach below became a
# 200,000 x 200,000 self-product (a 62.9GB commit, Event Viewer
# Resource-Exhaustion-Detector 2004) that took the desktop down with it.
# (impression_id, user_id) is unique on every population, including that one.
ROW_KEY = ["impression_id", "user_id"]

# Same flag/convention as src/build_pipeline.ipynb, src/bm25_retrieval.ipynb,
# src/evaluation_harness.ipynb.
BUILD_LARGE_ONLY = True
_DEFAULT_DATASETS = (
    ["ebnerd_large", "mind_large"]
    if BUILD_LARGE_ONLY
    else ["ebnerd", "ebnerd_small", "mind", "ebnerd_large", "mind_large"]
)
# FEATURE_DATASETS env var (comma-separated) restricts this run to a subset,
# same reasoning as EVAL_DATASETS in evaluation_harness.ipynb: run once per
# dataset rather than holding two large datasets' lookups in one process.
_env_datasets = os.environ.get("FEATURE_DATASETS")
DATASETS = _env_datasets.split(",") if _env_datasets else _DEFAULT_DATASETS


# The blind leaderboard populations are featurized by this same notebook
# (SPEC.md A2 Q5 #6). What differs is where their training-time inputs come
# from -- reranker.SUBMISSION_POPULATIONS maps each to the dataset the
# booster was fitted on -- and that they carry no click labels.
def is_submission_population(name: str) -> bool:
    return name in SUBMISSION_POPULATIONS


def catalog_dataset(name: str) -> str:
    """Dataset whose articles.parquet / article_embeddings.parquet `name` uses
    (ebnerd_testset has none of its own: its catalog is ebnerd_large's)."""
    return SUBMISSION_POPULATIONS.get(name, {}).get("catalog_dataset", name)


def train_dataset(name: str) -> str:
    """Dataset whose train split defines `popularity` for `name`."""
    return SUBMISSION_POPULATIONS.get(name, {}).get("train_dataset", name)


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


log_progress(f"feature_engineering started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY}, datasets={DATASETS})")


def scan_behaviors(dataset: str) -> pl.LazyFrame:
    """behaviors is never held resident: article_ids_inview alone is 5.47GB of
    ebnerd_large's 7.83GB frame, and everything here either needs a projected
    subset (counts, flags) or one 200k-impression chunk at a time
    (SPEC.md A2 Q1 #8)."""
    return pl.scan_parquet(feature_store[dataset]["behaviors_path"])


feature_store = {}
for name in DATASETS:
    feature_store[name] = {
        "articles": pl.read_parquet(
            DATA_DIR / catalog_dataset(name) / "articles.parquet", columns=["article_id", "category", "published_time"]
        ),
        "behaviors_path": DATA_DIR / name / "behaviors.parquet",
        "history": pl.read_parquet(
            DATA_DIR / name / "history.parquet",
            columns=["user_id", "article_id_sequence", "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence"],
        ),
    }
    log_progress(f"  {name}: loaded articles/history (behaviors read lazily)")

split_counts = {
    name: dict(
        zip(*scan_behaviors(name).group_by("split").len().collect().to_dict(as_series=False).values())
    )
    for name in DATASETS
}

# Per-dataset capability flags (SPEC.md A2 Q1 #2/#3) -- data facts derived
# from the real schema/null patterns, not hardcoded per dataset name, since
# these are exactly the columns whose nullness this feature set has to respect.
dataset_flags = {}
for name in DATASETS:
    history = feature_store[name]["history"]
    articles = feature_store[name]["articles"]
    n_rows, n_null_session = (
        scan_behaviors(name)
        .select(pl.len().alias("n"), pl.col("session_id").null_count().alias("n_null"))
        .collect()
        .row(0)
    )
    # Detected from the DATA (null counts), not the column dtype. The two
    # MIND tracks write the same semantically-absent columns with different
    # types -- `mind` uses Null, `mind_large` uses String -- while both are
    # 100% null, so a `dtype != pl.Null` test silently classified
    # `mind_large` as having real timestamps and dwell-time data. That is
    # wrong for MIND by construction (SPEC.md A2 Q1 #3) and would have
    # produced elapsed_time recency weights from nonexistent timestamps.
    n_users = history.height
    has_temporal = history["timestamp_sequence"].null_count() < n_users
    has_dwell = history["read_time_sequence"].null_count() < n_users
    dataset_flags[name] = {
        "recency_weight_basis": "elapsed_time" if has_temporal else "ordinal_proxy",
        "has_dwell_time": has_dwell,
        "has_session_data": n_null_session < n_rows,
        "has_freshness": articles["published_time"].null_count() < articles.height,
        # Absent (not null-filled) on the blind test populations: they are
        # the leaderboards' held-out labels. Everything downstream that
        # reads clicks branches on this flag.
        "has_click_labels": "article_ids_clicked" in scan_behaviors(name).collect_schema().names(),
    }

category_lookup = {
    name: dict(zip(feature_store[name]["articles"]["article_id"].to_list(), feature_store[name]["articles"]["category"].to_list()))
    for name in DATASETS
}
published_lookup = {
    name: dict(zip(feature_store[name]["articles"]["article_id"].to_list(), feature_store[name]["articles"]["published_time"].to_list()))
    for name in DATASETS
}

# The embeddings DataFrame is released as soon as the numpy lookup is built:
# holding both duplicates every vector (0.72GB Arrow + 0.38GB numpy at
# ebnerd_large), the same reason evaluation_harness.ipynb does `del
# embeddings_raw`.
#
# Converted in Arrow (`list.to_array(dim).to_numpy()`), not via
# `[np.asarray(v) for v in .to_list()]`: the comprehension materializes
# ~96M Python floats on the way to a 385MB matrix and measured a 3.5GB
# transient on top of a 7.5GB resident baseline for ebnerd_testset. The
# lookup values are float32 views into one block, bit-identical to what the
# comprehension produced (validated: `ebnerd` regenerates frame-equal).
# Same construction as retrieval.build_stage1_scorers (SPEC.md A2 Q2 #3).
embedding_lookup = {}
for name in DATASETS:
    _emb = pl.read_parquet(DATA_DIR / catalog_dataset(name) / "article_embeddings.parquet")
    _dim = len(_emb["embedding"][0])
    _mat = _emb["embedding"].list.to_array(_dim).to_numpy().astype(np.float32)
    embedding_lookup[name] = {aid: _mat[i] for i, aid in enumerate(_emb["article_id"].to_list())}
    del _emb, _mat


class HistoryStore:
    """Per-user history access that materializes one user's sequences at a
    time instead of every user's at once (SPEC.md A2 Q1 #8).

    A fully-materialized `user_id -> {ids, timestamps, read_times, scrolls}`
    dict costs 182 bytes per history element (measured with `tracemalloc` on
    ebnerd_small: 445MB for 2,560,542 elements). ebnerd_large holds
    131,918,897 history elements across 974,791 users, which projects to
    ~22.4GB -- more than this machine's total RAM, and the direct cause of a
    runaway-memory kill at 10.3GB resident on the first ebnerd_large attempt.
    Keeping only a `user_id -> row index` map (2.1MB per 18,827 users) and
    slicing a single row on demand (~58us) bounds memory by history *length*
    rather than by user *count*.

    The single-entry cache mirrors the BM25 adapter's last-user cache in
    `evaluation_harness.ipynb`. It only pays off when consecutive impressions
    share a user, which the source files no longer guarantee -- globally
    sorting them by user_id cost +3.7GB of peak memory, more than the ~5%
    throughput the cache was buying (SPEC.md A2 Q1 #8).
    """

    # Users absent from history.parquet (no pre-window clicks at all). Shared
    # and never mutated -- compute_impression_history_summary short-circuits
    # on the empty `ids` before reading any other key.
    _EMPTY = {"ids": [], "timestamps": None, "read_times": None, "scrolls": None}

    def __init__(self, history: pl.DataFrame, has_temporal: bool, has_dwell: bool):
        self._idx = {uid: i for i, uid in enumerate(history["user_id"].to_list())}
        self._ids = history["article_id_sequence"]
        self._timestamps = history["timestamp_sequence"]
        self._read_times = history["read_time_sequence"]
        self._scrolls = history["scroll_percentage_sequence"]
        # Passed in from dataset_flags rather than re-derived, so the store
        # and the flags can never disagree. An all-null column holds a
        # scalar None per row rather than a list, so it cannot be sliced and
        # .to_list()-ed like a populated one.
        self._has_temporal = has_temporal
        self._has_dwell = has_dwell
        self._cached_user = None
        self._cached_hist = None

    def __contains__(self, user_id) -> bool:
        return user_id in self._idx

    def __len__(self) -> int:
        return len(self._idx)

    def user_ids(self):
        return self._idx.keys()

    def items(self):
        for user_id in self._idx:
            yield user_id, self.get(user_id)

    def get(self, user_id) -> dict:
        if user_id == self._cached_user:
            return self._cached_hist
        i = self._idx.get(user_id)
        if i is None:
            return self._EMPTY
        # `_seq` guards a per-row null even on a column that has data
        # elsewhere -- a user with no recorded history is a null entry, not
        # an empty list, and must read the same as an absent user.
        def _seq(series, enabled):
            if not enabled:
                return None
            value = series[i]
            return None if value is None else value.to_list()

        ids = _seq(self._ids, True)
        hist = {
            "ids": ids if ids is not None else [],
            "timestamps": _seq(self._timestamps, self._has_temporal),
            "read_times": _seq(self._read_times, self._has_dwell),
            "scrolls": _seq(self._scrolls, self._has_dwell),
        }
        self._cached_user, self._cached_hist = user_id, hist
        return hist


history_lookup = {
    name: HistoryStore(
        feature_store[name]["history"],
        has_temporal=dataset_flags[name]["recency_weight_basis"] == "elapsed_time",
        has_dwell=dataset_flags[name]["has_dwell_time"],
    )
    for name in DATASETS
}

# popularity_lookup: train-split only, shared basis with Q4's novelty metric
# (SPEC.md A2 Q1 #7) via evaluation.train_popularity_lookup. Counts come from
# a columnar explode + group_by rather than a Python Counter over
# 10,384,901 materialized click lists.
popularity_lookup = {}
for name in DATASETS:
    train = train_dataset(name)
    counts_df = (
        pl.scan_parquet(DATA_DIR / train / "behaviors.parquet")
        .filter(pl.col("split") == "train")
        .select(pl.col("article_ids_clicked").explode(empty_as_null=True).alias("article_id"))
        .drop_nulls()
        .group_by("article_id")
        .len()
        .collect()
    )
    counts = dict(zip(counts_df["article_id"].to_list(), counts_df["len"].to_list()))
    if train == name:
        popularity_lookup[name] = train_popularity_lookup(
            feature_store[name]["articles"]["article_id"].to_list(), counts
        )
    else:
        # A submission population reuses the training dataset's lookup
        # verbatim, re-keyed into its own article namespace. The lookup is
        # built over the TRAINING catalog so the add-one smoothing
        # denominator is the one the booster saw; an article the population
        # adds after training (MINDlarge_test's test-only news) gets the
        # same smoothed-zero value a never-clicked training article gets.
        train_article_ids = pl.read_parquet(DATA_DIR / train / "articles.parquet", columns=["article_id"])["article_id"].to_list()
        train_lookup = train_popularity_lookup(train_article_ids, counts)
        smoothed_zero = 1.0 / (sum(counts.values()) + len(train_article_ids))
        rekeyed = {population_article_id(name, aid): pop for aid, pop in train_lookup.items()}
        popularity_lookup[name] = {
            aid: rekeyed.get(aid, smoothed_zero) for aid in feature_store[name]["articles"]["article_id"].to_list()
        }
        del train_article_ids, train_lookup, rekeyed
    del counts_df, counts

log_progress(f"lookups built for {DATASETS}")
dataset_flags

{'ebnerd_testset': {'recency_weight_basis': 'elapsed_time',
  'has_dwell_time': True,
  'has_session_data': True,
  'has_freshness': True,
  'has_click_labels': False}}

In [2]:
def test_setup():
    for name in DATASETS:
        store = history_lookup[name]
        history_user_ids = feature_store[name]["history"]["user_id"].to_list()
        assert set(store.user_ids()) <= set(history_user_ids)
        assert len(popularity_lookup[name]) == len(feature_store[name]["articles"])
        assert all(0.0 < p <= 1.0 for p in popularity_lookup[name].values())

        # HistoryStore must return exactly what a fully-materialized dict
        # would have, including the cache path (a second get() for the same
        # user returns the same content) and the absent-user path.
        probe_users = history_user_ids[:50]
        history_head = feature_store[name]["history"].head(50)
        expected_ids = history_head["article_id_sequence"].to_list()
        for uid, ids in zip(probe_users, expected_ids):
            assert store.get(uid)["ids"] == ids
            assert store.get(uid)["ids"] == ids  # cache hit returns the same thing
        assert store.get("__no_such_user__")["ids"] == []

    # known facts about these two specific datasets (SPEC.md A2 Q1 #2/#3),
    # only checked when actually in scope this run (not the case under
    # BUILD_LARGE_ONLY, where DATASETS is ["ebnerd_large", "mind_large"])
    if "ebnerd_small" in DATASETS:
        assert dataset_flags["ebnerd_small"] == {
            "recency_weight_basis": "elapsed_time",
            "has_dwell_time": True,
            "has_session_data": True,
            "has_freshness": True,
            "has_click_labels": True,
        }
    # Both MIND tracks must read as MIND regardless of how the pipeline
    # typed their empty columns: `mind` writes them as Null, `mind_large` as
    # String, both 100% null. A dtype-based check classified mind_large as
    # having real timestamps and dwell-time data; capability detection is
    # null-count-based precisely so that cannot happen (SPEC.md A2 Q1 #3).
    MIND_FLAGS = {
        "recency_weight_basis": "ordinal_proxy",
        "has_dwell_time": False,
        "has_session_data": False,
        "has_freshness": False,
        "has_click_labels": True,
    }
    for mind_track in ("mind", "mind_large"):
        if mind_track in DATASETS:
            assert dataset_flags[mind_track] == MIND_FLAGS, f"{mind_track}: {dataset_flags[mind_track]}"

    # Submission populations: no labels, and the re-keyed popularity must
    # actually resolve -- a namespace mismatch would silently give every
    # candidate the smoothed-zero value (a constant feature at serving time
    # where the booster leaned on it for 16-53% of its gain).
    for name in DATASETS:
        if not is_submission_population(name):
            continue
        assert dataset_flags[name]["has_click_labels"] is False, name
        pops = popularity_lookup[name]
        smoothed_zero = min(pops.values())
        n_hit = sum(1 for p in pops.values() if p > smoothed_zero)
        assert n_hit > 1_000, f"{name}: only {n_hit} population articles carry a train click count"
        assert all(aid.startswith(SUBMISSION_POPULATIONS[name]["article_prefix"][1]) for aid in list(pops)[:1000])

    # An all-null column must read as "no data" whatever its dtype -- the
    # exact divergence between the two MIND tracks, checked directly.
    for dtype in (pl.Null, pl.String):
        probe = pl.DataFrame({"seq": pl.Series([None, None], dtype=dtype)})
        assert probe["seq"].null_count() == probe.height


test_setup()
print("ok: lookups cover every article/user, popularity in (0,1], HistoryStore matches the materialized sequences (cached and uncached), and per-dataset capability flags match known EB-NeRD/MIND facts when in scope")

ok: lookups cover every article/user, popularity in (0,1], HistoryStore matches the materialized sequences (cached and uncached), and per-dataset capability flags match known EB-NeRD/MIND facts when in scope


## `weighted_mean_pool` regression test

Uniform weights must reduce to `mean_pool`'s exact output (SPEC.md A2 Q1
#5) -- checked before this is trusted anywhere downstream.

In [3]:
def test_weighted_mean_pool_matches_mean_pool():
    lookup = {"a": np.array([1.0, 2.0]), "b": np.array([3.0, 4.0]), "c": np.array([5.0, 6.0])}
    ids = ["a", "b", "c"]

    uniform = weighted_mean_pool(ids, [1.0, 1.0, 1.0], lookup)
    assert np.allclose(uniform, mean_pool(ids, lookup))

    all_on_a = weighted_mean_pool(ids, [1.0, 0.0, 0.0], lookup)
    assert np.allclose(all_on_a, lookup["a"])

    assert weighted_mean_pool(["z"], [1.0], lookup) is None

    # all-zero weights among matched ids -> falls back to unweighted mean,
    # not a division-by-zero crash or a silently undefined result
    zero_w = weighted_mean_pool(ids, [0.0, 0.0, 0.0], lookup)
    assert np.allclose(zero_w, mean_pool(ids, lookup))


test_weighted_mean_pool_matches_mean_pool()
print("ok: weighted_mean_pool with uniform weights is bit-identical to mean_pool; all-weight-on-one-item and all-zero-weight edge cases behave as documented")

ok: weighted_mean_pool with uniform weights is bit-identical to mean_pool; all-weight-on-one-item and all-zero-weight edge cases behave as documented


## Session features (vectorized)

`clicks_earlier_in_session`/`session_impressions_so_far` computed directly
from `behaviors` with `polars` window functions over `(user_id,
session_id)` ordered by `impression_time` -- no per-row Python loop, no
dependency on the history-based features below (SPEC.md A2 Q1 #6). Null for
MIND (`has_session_data = False`), not computed against a meaningless
single implicit session.

In [4]:
SESSION_SORT_KEYS = ["user_id", "session_id", "impression_time"]


def sink_parquet_atomic(lf: pl.LazyFrame, path: Path) -> None:
    """`sink_parquet` to a temporary path, then `os.replace` into place.

    A direct sink leaves a truncated file behind if the process is killed
    mid-write, and both callers below treat "the file exists" as "the file is
    complete" -- so a partial write would be silently accepted on the next
    run. Same write-then-rename discipline the chunk writes already use
    (`os.replace` is atomic on Windows and POSIX), applied to the two
    remaining non-atomic writes in this notebook.
    """
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    lf.sink_parquet(tmp_path)
    os.replace(tmp_path, path)


def session_features_path(dataset: str) -> Path:
    return CHECKPOINT_DIR / dataset / "_session_features.parquet"


def build_session_features(dataset: str) -> Path:
    """Compute `clicks_earlier_in_session`/`session_impressions_so_far` for
    every impression and persist them, keyed by ROW_KEY.

    Written to disk rather than kept resident: `generate_features` only needs
    them 200k impressions at a time, and at ebnerd_large's 24,630,275
    impressions the alternatives were both expensive -- a Python dict costs
    159 bytes/entry (3.65GB) and joining a resident frame into each split's
    source added +4.0GB of peak (SPEC.md A2 Q1 #8).
    """
    out_path = session_features_path(dataset)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return out_path

    if not dataset_flags[dataset]["has_session_data"]:
        sink_parquet_atomic(
            scan_behaviors(dataset).select(
                *ROW_KEY,
                pl.lit(None, dtype=pl.Int64).alias("clicks_earlier_in_session"),
                pl.lit(None, dtype=pl.Int64).alias("session_impressions_so_far"),
            ),
            out_path,
        )
        return out_path

    # Without labels the click running total has nothing to count -- the
    # column is nulled below rather than reported as zero -- but
    # session_impressions_so_far needs only the impression ordering, which
    # the blind populations do have.
    has_labels = dataset_flags[dataset]["has_click_labels"]
    n_clicked_expr = (
        pl.col("article_ids_clicked").list.len().alias("n_clicked") if has_labels
        else pl.lit(0, dtype=pl.UInt32).alias("n_clicked")
    )
    n = pl.col("n_clicked")
    # A group boundary is just "this row's (user, session) differs from the
    # previous row's" once sorted. Taking a GLOBAL cumulative sum/row count,
    # remembering its value at each group's first row (forward-filled down
    # the group) and subtracting gives the within-group running total
    # excluding the current row -- exactly what `.over()` computes, but
    # without polars' partitioned-window machinery, which cost +3.1GB of
    # peak memory and 3.4x the wall time at ebnerd_large scale
    # (SPEC.md A2 Q1 #8).
    #
    # ne_missing, not !=: `null != null` is null under three-valued logic
    # whereas `.over()` groups nulls together, so a plain != would split
    # every null-session row into its own group.
    is_new = (
        pl.col("user_id").ne_missing(pl.col("user_id").shift(1))
        | pl.col("session_id").ne_missing(pl.col("session_id").shift(1))
    ).fill_null(True)

    sink_parquet_atomic(
        scan_behaviors(dataset)
        .select(
            "impression_id", "user_id", "impression_time", "session_id",
            n_clicked_expr,
        )
        .sort(SESSION_SORT_KEYS)
        .with_columns(is_new.alias("_new"))
        .with_columns(
            (n.cum_sum() - n).alias("_cs_excl"),
            pl.int_range(pl.len(), dtype=pl.Int64).alias("_ri"),
        )
        .with_columns(
            pl.when(pl.col("_new")).then(pl.col("_cs_excl")).otherwise(None).forward_fill().alias("_cs_at_start"),
            pl.when(pl.col("_new")).then(pl.col("_ri")).otherwise(None).forward_fill().alias("_ri_at_start"),
        )
        .with_columns(
            (pl.col("_cs_excl") - pl.col("_cs_at_start")).alias("clicks_earlier_in_session"),
            (pl.col("_ri") - pl.col("_ri_at_start")).alias("session_impressions_so_far"),
        )
        .select(
            *ROW_KEY,
            (pl.col("clicks_earlier_in_session") if has_labels else pl.lit(None, dtype=pl.Int64)).alias("clicks_earlier_in_session"),
            "session_impressions_so_far",
        ),
        out_path,
    )
    return out_path


session_feature_paths = {name: build_session_features(name) for name in DATASETS}
log_progress(f"session features computed for {DATASETS}")
{name: pl.scan_parquet(p).select(pl.len()).collect().item() for name, p in session_feature_paths.items()}

{'ebnerd_testset': 13536710}

In [5]:
def test_session_features():
    for name in DATASETS:
        sf = pl.read_parquet(session_feature_paths[name])
        # Same row set, keyed by ROW_KEY: equal heights, unique keys on the
        # session side, and equal order-independent key digests (sum of the
        # 64-bit key hashes; a collision would need two different 13.5M-row
        # key sets to sum to the same value). Not an anti-join: hashing 13.5M
        # composite string keys as a join build side is a multi-GB step on
        # the blind populations, and this is a test cell.
        key_digest = pl.struct(ROW_KEY).hash(seed=0).cast(pl.UInt64).sum()
        n_behaviors, behaviors_digest = scan_behaviors(name).select(pl.len(), key_digest).collect(engine="streaming").row(0)
        assert sf.height == n_behaviors == sf.select(pl.struct(ROW_KEY).n_unique()).item()
        assert sf.select(key_digest).item() == behaviors_digest, f"{name}: session-feature keys differ from behaviors keys"
        if dataset_flags[name]["has_session_data"]:
            if dataset_flags[name]["has_click_labels"]:
                assert sf["clicks_earlier_in_session"].null_count() == 0
                assert (sf["clicks_earlier_in_session"] >= 0).all()
            else:
                assert sf["clicks_earlier_in_session"].null_count() == sf.height
            assert sf["session_impressions_so_far"].null_count() == 0
            assert (sf["session_impressions_so_far"] >= 0).all()
        else:
            assert sf["clicks_earlier_in_session"].null_count() == sf.height
            assert sf["session_impressions_so_far"].null_count() == sf.height
        del sf

    # The forward-fill formulation must agree exactly with the reference
    # partitioned-window (`.over()`) computation it replaced -- checked on a
    # hand-built toy covering the cases that actually differ: a multi-row
    # session, a session boundary, and (the one that caught a real bug) an
    # all-null session_id, where `!=` yields null but `.over()` groups nulls
    # together (SPEC.md A2 Q1 #6).
    toy = pl.DataFrame({
        "impression_id": ["i1", "i2", "i3", "i4", "n1", "n2"],
        "user_id": ["u1", "u1", "u1", "u1", "u9", "u9"],
        "session_id": ["s1", "s1", "s1", "s2", None, None],
        "impression_time": [1, 2, 3, 4, 1, 2],
        "n_clicked": [1, 2, 0, 5, 3, 4],
    })
    n = pl.col("n_clicked")
    expected = toy.sort(SESSION_SORT_KEYS).with_columns(
        (n.cum_sum().over(["user_id", "session_id"]) - n).alias("clicks_earlier_in_session"),
        (n.cum_count().over(["user_id", "session_id"]) - 1).alias("session_impressions_so_far"),
    ).select("impression_id", "clicks_earlier_in_session", "session_impressions_so_far").sort("impression_id")

    is_new = (
        pl.col("user_id").ne_missing(pl.col("user_id").shift(1))
        | pl.col("session_id").ne_missing(pl.col("session_id").shift(1))
    ).fill_null(True)
    actual = (
        toy.sort(SESSION_SORT_KEYS)
        .with_columns(is_new.alias("_new"))
        .with_columns((n.cum_sum() - n).alias("_cs_excl"), pl.int_range(pl.len(), dtype=pl.Int64).alias("_ri"))
        .with_columns(
            pl.when(pl.col("_new")).then(pl.col("_cs_excl")).otherwise(None).forward_fill().alias("_cs_at_start"),
            pl.when(pl.col("_new")).then(pl.col("_ri")).otherwise(None).forward_fill().alias("_ri_at_start"),
        )
        .with_columns(
            (pl.col("_cs_excl") - pl.col("_cs_at_start")).alias("clicks_earlier_in_session"),
            (pl.col("_ri") - pl.col("_ri_at_start")).alias("session_impressions_so_far"),
        )
        .select("impression_id", "clicks_earlier_in_session", "session_impressions_so_far").sort("impression_id")
    )
    assert expected.equals(actual), f"forward-fill diverged from .over():\n{expected}\n{actual}"

    # and the hand-computed values themselves
    got = dict(zip(actual["impression_id"].to_list(), zip(
        actual["clicks_earlier_in_session"].to_list(), actual["session_impressions_so_far"].to_list())))
    assert got["i1"] == (0, 0) and got["i2"] == (1, 1) and got["i3"] == (3, 2)
    assert got["i4"] == (0, 0)          # new session resets both
    assert got["n1"] == (0, 0) and got["n2"] == (3, 1)  # null session_id is one group, not two


test_session_features()
print("ok: session features complete for EB-NeRD, null for MIND, and the forward-fill formulation matches .over() including the all-null-session case")

ok: session features complete for EB-NeRD, null for MIND, and the forward-fill formulation matches .over() including the all-null-session case


## History-based features (per impression, recency-weighted)

Recency weight is relative to *this* impression's `impression_time`, not a
fixed per-user "now" (SPEC.md A2 Q1 #3) -- computed fresh per impression
from the per-user `history_lookup` built above.

In [6]:
def _cosine(a, b) -> float:
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def _weighted_avg_skip_none(values, weights):
    # Individual read_time/scroll_percentage entries can be null even within
    # a non-null history (16,510/18,827 ebnerd_small users have at least one
    # such entry -- a real EB-NeRD data fact, not a pipeline bug). Averaging
    # over only the non-null entries (with their matching weights) is the
    # correct behavior; None here (not 0.0 or a crash) means "this user has
    # no usable dwell-time signal at all in their history."
    pairs = [(v, w) for v, w in zip(values, weights) if v is not None]
    if not pairs:
        return None
    vals, ws = zip(*pairs)
    return float(np.average(vals, weights=ws))


def compute_recency_weights(hist: dict, impression_time, basis: str) -> np.ndarray:
    ids = hist["ids"]
    n = len(ids)
    if n == 0:
        return np.empty(0, dtype=np.float64)
    if basis == "elapsed_time":
        delta_hours = np.array([(impression_time - t).total_seconds() / 3600.0 for t in hist["timestamps"]])
        return np.exp(-np.log(2) * delta_hours / HALF_LIFE_HOURS)
    rank_from_recent = np.arange(n - 1, -1, -1, dtype=np.float64)  # last item (most recent) -> rank 0
    return np.exp(-np.log(2) * rank_from_recent / HALF_LIFE_CLICKS)


def compute_impression_history_summary(dataset: str, hist: dict, impression_time) -> dict:
    ids = hist["ids"]
    n = len(ids)
    if n == 0:
        return {
            "click_count": 0, "total_weight": 0.0, "cat_weights": {},
            "weighted_read_time": None, "weighted_scroll_percentage": None,
            "history_vector": None, "category_set": set(),
        }

    basis = dataset_flags[dataset]["recency_weight_basis"]
    cat_lookup = category_lookup[dataset]
    emb_lookup = embedding_lookup[dataset]
    weights = compute_recency_weights(hist, impression_time, basis)

    cat_weights: dict = {}
    for aid, w in zip(ids, weights):
        cat = cat_lookup.get(aid)
        if cat is not None:
            cat_weights[cat] = cat_weights.get(cat, 0.0) + float(w)

    read_times, scrolls = hist["read_times"], hist["scrolls"]
    weighted_read_time = _weighted_avg_skip_none(read_times, weights) if read_times is not None else None
    weighted_scroll = _weighted_avg_skip_none(scrolls, weights) if scrolls is not None else None

    return {
        "click_count": n,
        "total_weight": float(weights.sum()),
        "cat_weights": cat_weights,
        "weighted_read_time": weighted_read_time,
        "weighted_scroll_percentage": weighted_scroll,
        "history_vector": weighted_mean_pool(ids, weights.tolist(), emb_lookup),
        "category_set": {cat_lookup[aid] for aid in ids if aid in cat_lookup},
    }


def build_candidate_row(dataset: str, summary: dict, article_id: str, position: int, impression_time) -> dict:
    cat = category_lookup[dataset].get(article_id)
    total_w = summary["total_weight"]
    weighted_category_affinity = (summary["cat_weights"].get(cat, 0.0) / total_w) if total_w > 0 else 0.0

    history_vector = summary["history_vector"]
    candidate_vec = embedding_lookup[dataset].get(article_id)
    weighted_embedding_similarity = (
        _cosine(history_vector, candidate_vec) if history_vector is not None and candidate_vec is not None else None
    )

    published = published_lookup[dataset].get(article_id)
    freshness_hours = ((impression_time - published).total_seconds() / 3600.0) if published is not None else None

    return {
        "click_count": summary["click_count"],
        "weighted_category_affinity": weighted_category_affinity,
        "weighted_read_time": summary["weighted_read_time"],
        "weighted_scroll_percentage": summary["weighted_scroll_percentage"],
        "weighted_embedding_similarity": weighted_embedding_similarity,
        "popularity": popularity_lookup[dataset].get(article_id, 0.0),
        "freshness_hours": freshness_hours,
        "category_match": cat in summary["category_set"],
        "position_in_impression": position,
    }

In [7]:
def test_recency_weight_formulas():
    from datetime import datetime, timedelta, timezone as tz
    now = datetime(2024, 1, 10, tzinfo=tz.utc)

    hist = {"ids": ["h1", "h2"], "timestamps": [now - timedelta(hours=24), now - timedelta(hours=1)], "read_times": [10.0, 30.0], "scrolls": [50.0, 90.0]}
    weights = compute_recency_weights(hist, now, "elapsed_time")
    assert weights[1] > weights[0]  # 1h-old click weighted higher than 24h-old
    expected = np.exp(-np.log(2) * np.array([24.0, 1.0]) / HALF_LIFE_HOURS)
    assert np.allclose(weights, expected)
    assert 10.0 < np.average([10.0, 30.0], weights=weights) < 30.0

    ordinal_hist = {"ids": ["h1", "h2"], "timestamps": None, "read_times": None, "scrolls": None}
    ordinal_weights = compute_recency_weights(ordinal_hist, now, "ordinal_proxy")
    assert ordinal_weights[1] == 1.0  # most recent (last in sequence) -> rank 0 -> weight 1
    assert ordinal_weights[0] < ordinal_weights[1]

    assert len(compute_recency_weights({"ids": [], "timestamps": [], "read_times": [], "scrolls": []}, now, "elapsed_time")) == 0


test_recency_weight_formulas()
print("ok: recency-weight formulas match hand-computed values for both bases; more recent clicks weighted higher")

ok: recency-weight formulas match hand-computed values for both bases; more recent clicks weighted higher


## Anti-gaming (Q9): recompute-from-truncated-input

SPEC.md A2 Q1 #9. Checked against real data, before the expensive full
feature table is generated below.

In [8]:
def test_no_future_leakage_in_features():
    for name in DATASETS:
        if dataset_flags[name]["recency_weight_basis"] != "elapsed_time":
            continue  # ordinal proxy has no timestamps to check against impression_time
        sample = scan_behaviors(name).filter(pl.col("split").is_in(["val", "test"])).select("user_id", "impression_time").head(200).collect()
        for row in sample.iter_rows(named=True):
            hist = history_lookup[name].get(row["user_id"])
            if not hist or not hist["ids"]:
                continue
            assert all(t < row["impression_time"] for t in hist["timestamps"]), (
                f"{name}: a history timestamp is not strictly before impression_time"
            )

    # Recompute-from-truncated-input: truncate one real user's history to a
    # single-item prefix (picking a user whose first read_time/scroll entry
    # is actually non-null -- individual entries can be null even within a
    # non-null history, see _weighted_avg_skip_none) and confirm the summary
    # computed from that truncated input matches a hand-derived value for
    # exactly that prefix. Needs an elapsed_time-basis dataset in scope --
    # skipped (not failed) on a MIND-only run, where there is no timestamp
    # data at all to run this specific check against; the leakage check
    # above still runs for every dataset regardless.
    target = next((n for n in DATASETS if dataset_flags[n]["recency_weight_basis"] == "elapsed_time"), None)
    if target is None:
        print("  (recompute-from-truncated-input check skipped: no elapsed_time-basis dataset in scope this run)")
        return
    user_id, hist = next(
        (uid, h) for uid, h in history_lookup[target].items()
        if len(h["ids"]) >= 2 and h["read_times"][0] is not None and h["scrolls"][0] is not None
    )
    truncated = {
        "ids": hist["ids"][:1],
        "timestamps": hist["timestamps"][:1],
        "read_times": hist["read_times"][:1],
        "scrolls": hist["scrolls"][:1],
    }
    fake_impression_time = hist["timestamps"][1]
    summary = compute_impression_history_summary(target, truncated, fake_impression_time)

    assert summary["click_count"] == 1
    # a weighted average of a single point equals that point's own raw value
    # up to floating-point rounding (x*w/w is not bit-exact for arbitrary
    # x, w in IEEE 754 -- a tolerance, not exact equality, is the correct check)
    assert abs(summary["weighted_read_time"] - truncated["read_times"][0]) < 1e-9
    assert abs(summary["weighted_scroll_percentage"] - truncated["scrolls"][0]) < 1e-9


test_no_future_leakage_in_features()
print("ok: history timestamps consumed are always strictly before impression_time; recompute-from-truncated-input matches by hand (or skipped where no such dataset is in scope)")

ok: history timestamps consumed are always strictly before impression_time; recompute-from-truncated-input matches by hand (or skipped where no such dataset is in scope)


## Feature table generation (chunked, checkpointed)

Full `val`/`test`, a capped seeded sample of `train` (SPEC.md A2 Q1 #9).
Same chunked, atomically-written, resumable-on-crash discipline as
`evaluate_ranking`/`generate_predictions`.

In [9]:
SOURCE_COLUMNS = ["impression_id", "user_id", "impression_time", "article_ids_inview", "article_ids_clicked"]


def source_columns(dataset: str) -> list[str]:
    return SOURCE_COLUMNS if dataset_flags[dataset]["has_click_labels"] else SOURCE_COLUMNS[:-1]


def build_split_source(dataset: str, split: str, tmp_path: Path) -> int:
    """Stream this split's rows (train-sampled) into a temporary parquet and
    return its row count.

    Deliberately a filter-and-sink only -- no join, no sort. Both were
    measured on ebnerd_large's 12,566,385-row test split and both are
    pipeline breakers that materialize the whole split: sorting by `user_id`
    cost +3.7GB of peak and joining the 24,630,275-row session frame +4.0GB,
    against a 7.48GB baseline for the plain filter+sink (SPEC.md A2 Q1 #8).
    Session values are attached per chunk instead, and dropping the user_id
    ordering costs only the HistoryStore cache (~58us per impression, ~5%).
    """
    if is_submission_population(dataset):
        # The blind populations are written in (user_id, impression_id)
        # order instead -- an external sort, bounded in memory (see
        # reranker.write_user_sorted_source) -- because
        # reranker_submission.ipynb scores Stage-1 over the same order and
        # aligns its chunks to this file's by position. The order is
        # user-contiguous so the BM25 adapter's one-entry cache pays off
        # there. Every row of these files is the test split.
        assert set(split_counts[dataset]) == {"test"} and split == "test", (dataset, split_counts[dataset])
        return write_user_sorted_source(feature_store[dataset]["behaviors_path"], tmp_path, source_columns(dataset))

    lf = pl.scan_parquet(feature_store[dataset]["behaviors_path"]).filter(pl.col("split") == split)

    n_train = split_counts[dataset].get("train", 0)
    if split == "train" and n_train > TRAIN_SAMPLE_CAP:
        keep = pl.Series("_ri", range(n_train), dtype=pl.UInt32).sample(n=TRAIN_SAMPLE_CAP, seed=TRAIN_SAMPLE_SEED)
        lf = lf.with_row_index("_ri").filter(pl.col("_ri").is_in(keep.implode())).drop("_ri")

    sink_parquet_atomic(lf.select(source_columns(dataset)), tmp_path)
    return pl.scan_parquet(tmp_path).select(pl.len()).collect().item()


def generate_features(dataset: str) -> Path:
    final_ckpt = DATA_DIR / dataset / "reranker_features.parquet"
    metrics_path = DATA_DIR / dataset / "feature_metrics.json"
    if final_ckpt.exists() and metrics_path.exists():
        log_progress(f"{dataset}: reranker_features.parquet already exists, skipping")
        return final_ckpt

    hist_lookup = history_lookup[dataset]
    sess_path = session_feature_paths[dataset]
    has_labels = dataset_flags[dataset]["has_click_labels"]

    chunk_dir = CHECKPOINT_DIR / dataset
    chunk_dir.mkdir(parents=True, exist_ok=True)

    chunk_paths = []
    split_row_counts = {}
    split_available_counts = {}
    for split in SPLITS:
        n_available = split_counts[dataset].get(split, 0)
        split_available_counts[split] = n_available
        n_impressions = min(n_available, TRAIN_SAMPLE_CAP) if split == "train" else n_available
        split_row_counts[split] = n_impressions
        if n_impressions == 0:
            continue  # the blind populations have a test split only

        chunk_bounds = list(range(0, n_impressions, CHUNK_SIZE)) + [n_impressions]
        split_chunk_paths = [chunk_dir / f"{split}_chunk_{c:03d}.parquet" for c in range(len(chunk_bounds) - 1)]
        chunk_paths.extend(split_chunk_paths)
        n_chunks = max(1, len(split_chunk_paths))

        # An already-complete split needs no source file at all on a resume.
        if all(p.exists() for p in split_chunk_paths):
            log_progress(f"  {dataset}/{split}: all {n_chunks} chunks already checkpointed, skipping")
            continue

        tmp_source = chunk_dir / f"_source_{split}.parquet"
        built = build_split_source(dataset, split, tmp_source)
        assert built == n_impressions, f"{dataset}/{split}: source has {built} rows, expected {n_impressions}"
        log_progress(f"  {dataset}/{split}: streamed source ({built} rows)")

        for c, chunk_path in enumerate(split_chunk_paths):
            start, end = chunk_bounds[c], chunk_bounds[c + 1]
            if chunk_path.exists():
                continue

            # Arrow -> Python conversion happens one chunk at a time, never
            # for a whole split at once. article_ids_inview averages ~11.9
            # article-id strings per impression and the conversion interns
            # nothing across rows (same effect documented for top10_ids in
            # evaluate_ranking), so a whole-split conversion would
            # materialize ~150M Python strings (~9GB) for ebnerd_large's
            # test split. Per chunk it is ~2.4M strings, freed as soon as
            # the chunk parquet is written (SPEC.md A2 Q1 #8).
            #
            # Session values are attached here, against this 200k-row slice,
            # in two steps that keep the SMALL side as the hash-table side.
            # polars builds the table on the right-hand side of a left join,
            # so `slice.join(session_file, how="left")` hashed the whole
            # session table -- tolerable at ebnerd_large on one string key,
            # but on ROW_KEY over ebnerd_testset's 13.5M rows it exceeded 6GB
            # per chunk. A semi-join with the slice on the right first cuts
            # the session table down to this chunk's rows (0.3s, <1GB), and
            # the left join is then 200k x 200k. On ROW_KEY, never
            # impression_id alone -- see ROW_KEY's comment.
            chunk_source = pl.scan_parquet(tmp_source).slice(start, end - start).collect()
            chunk_sessions = (
                pl.scan_parquet(sess_path)
                .join(chunk_source.lazy().select(ROW_KEY), on=ROW_KEY, how="semi")
                .collect()
            )
            chunk_behaviors = chunk_source.join(chunk_sessions, on=ROW_KEY, how="left")
            assert chunk_behaviors.height == chunk_source.height, (dataset, split, c)
            del chunk_source, chunk_sessions
            impression_ids = chunk_behaviors["impression_id"].to_list()
            user_ids = chunk_behaviors["user_id"].to_list()
            impression_times = chunk_behaviors["impression_time"].to_list()
            inviews = chunk_behaviors["article_ids_inview"].to_list()
            clickeds = chunk_behaviors["article_ids_clicked"].to_list() if has_labels else None
            clicks_earlier_col = chunk_behaviors["clicks_earlier_in_session"].to_list()
            sessions_so_far_col = chunk_behaviors["session_impressions_so_far"].to_list()
            del chunk_behaviors

            cols = {k: [] for k in [
                "impression_id", "user_id", "article_id", "clicked", "click_count",
                "weighted_category_affinity", "weighted_read_time", "weighted_scroll_percentage",
                "weighted_embedding_similarity", "position_in_impression", "clicks_earlier_in_session",
                "session_impressions_so_far", "popularity", "freshness_hours", "category_match",
            ]}

            for local_i in range(end - start):
                impression_id = impression_ids[local_i]
                user_id = user_ids[local_i]
                impression_time = impression_times[local_i]
                inview_ids = inviews[local_i]
                clicked_set = set(clickeds[local_i]) if has_labels else None
                clicks_earlier = clicks_earlier_col[local_i]
                sessions_so_far = sessions_so_far_col[local_i]

                hist = hist_lookup.get(user_id)
                summary = compute_impression_history_summary(dataset, hist, impression_time)

                for pos, aid in enumerate(inview_ids, start=1):
                    row = build_candidate_row(dataset, summary, aid, pos, impression_time)
                    cols["impression_id"].append(impression_id)
                    cols["user_id"].append(user_id)
                    cols["article_id"].append(aid)
                    cols["clicked"].append((aid in clicked_set) if has_labels else None)
                    cols["click_count"].append(row["click_count"])
                    cols["weighted_category_affinity"].append(row["weighted_category_affinity"])
                    cols["weighted_read_time"].append(row["weighted_read_time"])
                    cols["weighted_scroll_percentage"].append(row["weighted_scroll_percentage"])
                    cols["weighted_embedding_similarity"].append(row["weighted_embedding_similarity"])
                    cols["position_in_impression"].append(row["position_in_impression"])
                    cols["clicks_earlier_in_session"].append(clicks_earlier)
                    cols["session_impressions_so_far"].append(sessions_so_far)
                    cols["popularity"].append(row["popularity"])
                    cols["freshness_hours"].append(row["freshness_hours"])
                    cols["category_match"].append(row["category_match"])

                if (start + local_i + 1) % 20_000 == 0:
                    log_progress(f"    {dataset}/{split}: {start + local_i + 1}/{n_impressions} impressions featurized")

            size = len(cols["impression_id"])
            chunk_df = pl.DataFrame({
                **cols,
                "dataset": [dataset] * size,
                "split": [split] * size,
            }).with_columns(pl.col("clicked").cast(pl.Boolean))  # all-None -> Null dtype otherwise
            tmp_path = chunk_path.with_suffix(".parquet.tmp")
            chunk_df.write_parquet(tmp_path)
            os.replace(tmp_path, chunk_path)
            log_progress(f"  {dataset}/{split}: chunk {c + 1}/{n_chunks} checkpointed ({start}-{end})")
            del cols, chunk_df, impression_ids, user_ids, impression_times, inviews, clickeds

        tmp_source.unlink()

    # Streamed concat: at ebnerd_large's ~180M-row scale, reading every chunk
    # eagerly and holding them all before the write would defeat the
    # per-chunk memory bound above.
    tmp_final = final_ckpt.with_suffix(".parquet.tmp")
    pl.concat([pl.scan_parquet(p) for p in chunk_paths]).sink_parquet(tmp_final)
    os.replace(tmp_final, final_ckpt)
    n_rows = pl.scan_parquet(final_ckpt).select(pl.len()).collect().item()

    metrics = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        **dataset_flags[dataset],
        "half_life_hours": HALF_LIFE_HOURS,
        "half_life_clicks": HALF_LIFE_CLICKS,
        "train_sample_cap": TRAIN_SAMPLE_CAP,
        "train_sample_seed": TRAIN_SAMPLE_SEED,
        "n_rows": n_rows,
        "n_impressions_by_split": split_row_counts,
        "n_impressions_available_by_split": split_available_counts,
        "train_dataset": train_dataset(dataset),
        "catalog_dataset": catalog_dataset(dataset),
        "row_order": "user_id, impression_id" if is_submission_population(dataset) else "source file order",
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))

    shutil.rmtree(chunk_dir)
    log_progress(f"{dataset}: reranker_features.parquet + feature_metrics.json written ({n_rows} rows)")
    return final_ckpt


feature_paths = {name: generate_features(name) for name in DATASETS}
feature_paths

{'ebnerd_testset': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd_testset/reranker_features.parquet')}

In [10]:
def test_feature_table():
    import pyarrow.parquet as pq

    for name in DATASETS:
        # One bounded pass over the table, one parquet batch at a time. Every
        # whole-table operation here -- `pl.all().null_count()`, a semi-join
        # to pull 200 sample impressions, collecting the two key columns --
        # materialized the 206M-row blind tables under the in-memory engine
        # and killed the kernel (three 60GB commits recorded by Windows'
        # Resource-Exhaustion-Detector), so the checks are accumulated
        # per batch instead. (SPEC.md A2 Q1 #8, A2 Q5 #6.)
        flags = dataset_flags[name]
        pf = pq.ParquetFile(feature_paths[name])
        columns = pf.schema_arrow.names
        n_rows = 0
        nulls = {c: 0 for c in columns}
        bounds = {"pop_min": 1.0, "pop_max": 0.0, "aff_min": 0.0, "aff_max": 0.0, "pos_min": 1, "cc_min": 0}
        sample_keys = None      # first 200 distinct (impression_id, user_id) keys
        sample_rows = []        # their feature rows
        n_runs, prev_user, prev_key = 0, None, None
        for batch in pf.iter_batches(batch_size=1_000_000):
            b = pl.from_arrow(batch)
            n_rows += b.height
            for c, v in zip(columns, b.null_count().row(0)):
                nulls[c] += v
            agg = b.select(
                pl.col("popularity").min().alias("pop_min"), pl.col("popularity").max().alias("pop_max"),
                pl.col("weighted_category_affinity").min().alias("aff_min"), pl.col("weighted_category_affinity").max().alias("aff_max"),
                pl.col("position_in_impression").min().alias("pos_min"), pl.col("click_count").min().alias("cc_min"),
            ).row(0, named=True)
            for k in ("pop_min", "aff_min", "pos_min", "cc_min"):
                bounds[k] = min(bounds[k], agg[k])
            for k in ("pop_max", "aff_max"):
                bounds[k] = max(bounds[k], agg[k])
            if sample_keys is None:
                sample_keys = b.select(ROW_KEY).unique(maintain_order=True).head(200)
            sample_rows.append(b.join(sample_keys, on=ROW_KEY, how="semi"))
            if is_submission_population(name):
                # Row order is the contract reranker_submission.ipynb aligns
                # to: (user_id, impression_id)-sorted, impressions contiguous,
                # the full population present exactly once.
                users = b["user_id"]
                assert users.is_sorted(), f"{name}: rows are not user-sorted"
                assert prev_user is None or prev_user <= users[0], f"{name}: user order breaks at a batch boundary"
                keys = b.select(pl.struct(ROW_KEY).alias("k"))["k"]
                runs = b.select(pl.struct(ROW_KEY).rle_id().max() + 1).item()
                if prev_key is not None and prev_key == keys[0]:
                    runs -= 1  # an impression straddling the boundary is one run, not two
                n_runs += runs
                prev_user, prev_key = users[-1], keys[-1]
            del b

        never_null_cols = ["impression_id", "user_id", "article_id", "split", "click_count", "popularity", "position_in_impression", "category_match"]
        if flags["has_click_labels"]:
            never_null_cols.append("clicked")
        assert all(nulls[c] == 0 for c in never_null_cols), {c: nulls[c] for c in never_null_cols if nulls[c]}
        if not flags["has_click_labels"]:
            assert nulls["clicked"] == n_rows and pf.schema_arrow.field("clicked").type == "bool"

        assert 0.0 <= bounds["pop_min"] and bounds["pop_max"] <= 1.0
        # weighted_category_affinity = cat_weight_subset_sum / total_weight_sum
        # -- a ratio of two sums built from the same terms in different
        # groupings, which floating-point summation can push a hair above
        # 1.0 (observed max 1.0000000000000004 on ebnerd_small, 114/5.5M
        # rows) -- a tolerance, not exact [0,1], is the correct bound.
        assert -1e-9 <= bounds["aff_min"] and bounds["aff_max"] <= 1.0 + 1e-9
        assert bounds["pos_min"] >= 1 and bounds["cc_min"] >= 0

        if not flags["has_dwell_time"]:
            assert nulls["weighted_read_time"] == n_rows and nulls["weighted_scroll_percentage"] == n_rows
        if not flags["has_session_data"]:
            assert nulls["clicks_earlier_in_session"] == n_rows and nulls["session_impressions_so_far"] == n_rows
        else:
            assert nulls["session_impressions_so_far"] == 0
            assert nulls["clicks_earlier_in_session"] == (0 if flags["has_click_labels"] else n_rows)
        if not flags["has_freshness"]:
            assert nulls["freshness_hours"] == n_rows

        # Cross-check the sample impressions against the raw behaviors
        # parquet -- scanned for just those rows (semi-join with the 200 keys
        # as the build side) rather than materializing every impression's
        # inview/clicked lists. Keyed by ROW_KEY: ebnerd_testset's
        # beyond-accuracy rows share impression_id 0 across 200,000 users.
        sample = pl.concat(sample_rows)
        raw_cols = ROW_KEY + ["article_ids_inview"] + (["article_ids_clicked"] if flags["has_click_labels"] else [])
        raw = (
            pl.scan_parquet(feature_store[name]["behaviors_path"])
            .join(sample_keys.lazy(), on=ROW_KEY, how="semi")
            .select(raw_cols)
            .collect(engine="streaming")
        )
        raw_keys = list(zip(raw["impression_id"].to_list(), raw["user_id"].to_list()))
        inview_lookup = dict(zip(raw_keys, raw["article_ids_inview"].to_list()))
        clicked_lookup = dict(zip(raw_keys, raw["article_ids_clicked"].to_list())) if flags["has_click_labels"] else None
        for imp_id, uid in zip(sample_keys["impression_id"].to_list(), sample_keys["user_id"].to_list()):
            rows = sample.filter((pl.col("impression_id") == imp_id) & (pl.col("user_id") == uid))
            assert set(rows["article_id"].to_list()) == set(inview_lookup[(imp_id, uid)])
            if clicked_lookup is not None:
                assert set(rows.filter(pl.col("clicked"))["article_id"].to_list()) == set(clicked_lookup[(imp_id, uid)])
            assert sorted(rows["position_in_impression"].to_list()) == list(range(1, rows.height + 1))

        train_impressions_used = (
            pl.scan_parquet(feature_paths[name]).filter(pl.col("split") == "train")
            .select(pl.col("impression_id").n_unique()).collect(engine="streaming").item()
        )
        raw_train_count = split_counts[name].get("train", 0)
        assert train_impressions_used == min(raw_train_count, TRAIN_SAMPLE_CAP)

        if is_submission_population(name):
            # runs == impressions <=> every (impression, user) is contiguous
            # and appears exactly once; sorted order makes both hold together.
            assert n_runs == split_counts[name]["test"], (name, n_runs, split_counts[name]["test"])


test_feature_table()
print("ok: reranker_features.parquet matches article_ids_inview/article_ids_clicked exactly on a sample, respects per-dataset nullability, honors the train-sample cap, and is user-sorted for the blind populations")


ok: reranker_features.parquet matches article_ids_inview/article_ids_clicked exactly on a sample, respects per-dataset nullability, honors the train-sample cap, and is user-sorted for the blind populations


# Manual Verification Complete